In [ ]:
# 🛠️ Gemma 4 Autonomous Developer Agent: Calibrated Single-Agent Architecture

> An optimized, budget-calibrated autonomous developer agent for the **Google - The Gemma 4 Developer Agent Competition**.
> Engineered with empirically tuned execution limits, focused target localization, and strict anti-pattern guards.

---

### 📌 Key Strategic Insights & Design Decisions

Standard baseline submissions frequently suffer from two critical pitfalls:
1. **Runaway Exploration**: Spending 40+ tool calls wandering across unrelated repository files without ever making a source edit.
2. **Evaluation Timeouts**: Executing broad `pytest` commands across entire suites, taking several minutes and exhausting per-task or global budgets.

This notebook addresses these failure modes through rigorous operational limits and strict prompt directives:

| Dimension | Default Starter | This Implementation | Rationale |
| :--- | :--- | :--- | :--- |
| **Time Budget** (`max_time_minutes`) | `4.0` min (or uncapped) | **`8.0` min** | Gives hard tasks sufficient runway to perform compile-checks and targeted assertions. |
| **Tool Call Budget** (`max_tool_calls`) | `50 – 100` calls | **`28` calls** | Truncates unproductive rabbit holes. Runs that don't edit by turn 12 almost never resolve. |
| **Command Timeout** (`timeout_seconds`) | `60 – 120` sec | **`240` sec** | Accommodates heavy repos (e.g. Requests) during evaluation verification without timing out. |
| **Max Output Tokens** (`max_output_tokens`) | `16,384` tokens | **`8,192` tokens** | Prevents prompt + output from exceeding the strict 32k token vLLM context window limit. |
| **Reasoning Mode** | Variable | **Thinking Budget 4,096** | Enables Gemma 4 reasoning while reserving sufficient generation budget. |
| **Targeted Testing** | General pytest | **Filtered pytest (`-k`)** | Strictly forbids whole-repo test suites; enforces targeted single-test verification. |

---

### 🚀 Pipeline Overview

1. **Declarative Agent Manifest** (`agent.yaml`): Declares root model and tool registry bindings.
2. **Inference & Sampling Profile** (`configs/sampling.yaml`): Calibrates temperature, top-p, output tokens, and thinking budgets.
3. **Execution Guardrails** (`eval_config.yaml`): Sets hard per-task operational ceilings.
4. **Structured Engineering Prompt** (`prompts/system.md`): Concise instructions enforcing localization, direct editing, and rapid submission.
5. **Hermetic Packaging**: Validates bundle structure and packages directly into `submission.zip`.


In [ ]:
# ==============================================================================
# 1. Imports & Output Path Resolution
# ==============================================================================
import os
import sys
import zipfile
from pathlib import Path

# Resolve output working directory for Kaggle or local environments
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
SUBMISSION_PATH = OUTPUT_DIR / "submission.zip"

print(f"Target Submission Path: {SUBMISSION_PATH}")


In [ ]:
# ==============================================================================
# 2. Agent & Evaluation Configurations
# ==============================================================================

# Core Agent Specification (ADK Declarative Schema)
AGENT_YAML = """name: swe_agent
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph
generate_content_config: !include configs/sampling.yaml
"""

# Generation & Sampling Profile
SAMPLING_YAML = """temperature: 0.2
top_p: 0.95
max_output_tokens: 8192
thinking_config:
  thinking_budget: 4096
  include_thoughts: true
"""

# Execution & Evaluation Budget
EVAL_CONFIG_YAML = """evaluation:
  timeout_seconds: 240
  max_tool_calls: 28
  max_time_minutes: 8
  max_turns: 80
"""


In [ ]:
# ==============================================================================
# 3. System Prompt Engineering
# ==============================================================================

SYSTEM_PROMPT = """You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.

## Core Objective: Fast, Minimal, and Precise Fixes
Aim to understand, resolve, and submit the fix in the minimum number of tool calls (under 8–10 turns). Move directly from the problem statement to the relevant files, apply the solution, verify with a targeted test, and submit.

## Workflow

### 1. Identify Target Files Immediately
- Extract filenames, functions, classes, CLI subcommands, or error messages directly from the problem statement.
- Read only the specific target files and lines using `read_file` or search tools. Do not wander across unrelated files.
- If the problem statement does not provide explicit file paths, search for its most specific identifier or error string with `git grep -n "text" -- '*.py' | head -20`.
- Do not use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`: they only accept exact fully-qualified ids and usually return nothing.

### 2. Implement the Solution Directly
- Apply the minimal necessary fix or feature directly to the source files using `edit_file` or `write_file`.
- Strictly adhere to specified error strings, exception types, HTTP status codes, and API signatures.
- For documentation code tasks (e.g. FastAPI), edit executable code under `docs_src/`.

### 3. Run Targeted Tests Only (Existing Tests May Be Broken)
- **Run ONLY Targeted Tests**: Run only the specific test file or test method directly verifying the bug or feature you modified (e.g. `pytest tests/test_target.py -k test_feature` or `python3 -m unittest tests.test_target`).
- **Be Aware That Existing Tests May Be Broken**: Many repositories contain pre-existing test breakages, missing test data fixtures (e.g. `/test_data`), or environment import errors unrelated to your task.
- **Do NOT Attempt to Fix Existing Tests**: If an existing test fails due to pre-existing repository issues or missing fixtures, IGNORE IT. Never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.
- **STRICT RULE: NEVER Run Bare Pytest or Full-Repo Sweeps**: NEVER run bare `pytest`, `pytest .`, `python3 -m unittest discover`, or full-repo test suites without specifying a target file. Full test suites take several minutes, cause catastrophic timeouts, and exhaust your turn and time budgets.
- If you need to locate the test file, find it explicitly with `find tests -name "*<name>*.py"` instead of running the test runner across the repo.

### 4. Immediate Patch Submission
- Once your targeted test passes:
  1. Call `submit_patch` immediately.
  2. Verify `patch_size > 0` and `files_changed > 0`.
  3. Output a short summary of the fix to end the session.

## Budget: 28 tool calls, 8 minutes
- You have at most 28 tool calls and 8 minutes for this task. When either runs out, the working tree is graded as it is, so an edit made in time counts even if you never reach `submit_patch`.
- Use at most about 10 tool calls to find and read the code.
- By your 12th tool call you must have edited a source file. If you have not, stop exploring and make your best-guess edit in the most likely place now. Runs that are still only reading after 12 calls almost never succeed; more reading does not help.
- After your edit: one compile check or one targeted test, a correction if needed, then `submit_patch`. Do not start new exploration after your 20th tool call.

## Anti-Patterns to Avoid
- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.
- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) — always specify the exact test file path.
- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** — your task is strictly to implement the fix for the reported issue in source code.
- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed. If `ModuleNotFoundError` occurs during test runs, focus on fixing code under `/workspace`, not looking for missing system packages.
- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.
- Do NOT refactor or reformat unrelated functions or files.
- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications. Concluding that the codebase is already clean without making changes is an anti-pattern.
"""


In [ ]:
# ==============================================================================
# 4. Bundle Packaging & Validation
# ==============================================================================

SUBMISSION_FILES = {
    "agent.yaml": AGENT_YAML,
    "configs/sampling.yaml": SAMPLING_YAML,
    "eval_config.yaml": EVAL_CONFIG_YAML,
    "prompts/system.md": SYSTEM_PROMPT,
}

# Remove pre-existing archive if present
if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()

# Build compressed archive
with zipfile.ZipFile(SUBMISSION_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for filename, content in SUBMISSION_FILES.items():
        zf.writestr(filename, content)
        print(f"  + Added: {filename} ({len(content.encode('utf-8')):,} bytes)")

# Perform integrity checks
with zipfile.ZipFile(SUBMISSION_PATH, "r") as zf:
    assert zf.testzip() is None, "ZIP integrity test failed!"
    manifest = zf.namelist()
    assert "agent.yaml" in manifest, "Missing root agent.yaml!"
    assert "eval_config.yaml" in manifest, "Missing eval_config.yaml!"
    print("\nVerification Passed:")
    print(f"  Archive Size: {SUBMISSION_PATH.stat().st_size:,} bytes")
    print(f"  Total Files : {len(manifest)}")

print("\n[SUCCESS] submission.zip is ready for submission.")


In [ ]:
---
### 📋 How to Submit

1. Click **"Save Version"** in the top-right corner of the Kaggle notebook interface.
2. Select **"Save & Run All (Commit)"**.
3. Once the commit finishes, navigate to the **"Output"** tab on the right sidebar.
4. Locate `submission.zip` and click **"Submit"** to evaluate on the competition leaderboard.

*Good luck with your submissions!*
